# 4 – Lista de genes por eje biológico

Tabla suplementaria: genes curados por eje vs presencia en el grafo.

In [ ]:
import pandas as pd
import numpy as np

# ── Cargar genes del grafo ──
df_graph = pd.read_csv("./Results/genes_grafo.csv")
gene_col = next((c for c in ["gene", "Gene", "symbol"] if c in df_graph.columns), None)
assert gene_col, f"No gene column found in {df_graph.columns.tolist()}"
graph_genes = set(df_graph[gene_col].astype(str).str.strip().str.upper())
print(f"Genes únicos en el grafo: {len(graph_genes)}")

# ── Ejes curados ──
AXES = {
    "Luminal Hormonal": ["ESR1","PGR","GATA3","GREB1","ESR2","PHLDA1","LRIG1","RXRA"],
    "Cell Cycle Mitotic": ["AURKA","AURKB","BUB1","CDC7","CDC23","CDCA3","CDK1","CDK2",
                           "CEP55","E2F1","FOXM1","PLK1","TTK","ZWINT"],
    "HER2 RTK MAPK": ["ERBB2","EGFR","IGF1R","MET","PDGFRB","AKT1","MAP2K1","MAP2K2",
                       "MAPK1","MAPK3","MAPK14","PLCG1","SRC","LYN"],
    "Basal Plasticity TNBC": ["KRT6A","KRT16","SOX10","VGLL1","VGLL3","EGFR","EMP1","MSLN"],
    "Immune Lymphoid Signaling": ["CD79A","CXCL9","DEF6","GRAP2","HCK","JAK3","ITPKB","PAX5",
                                  "SYK","S1PR1","SOCS1","TRAF1","TRAF2","ZBP1"],
    "DNA Damage p53 Checkpoint": ["ATM","ERCC3","FANCG","KAT5","PRKDC","TP53","MDM2","PTEN",
                                  "DAPK1","STK11","SIRT1"],
    "Adhesion Cytoskeleton Invasion": ["ABI2","FLNA","ITGB1","ITGB1BP1","MMP2","PTK2","RHOA",
                                       "ROCK1","LASP1","NCK1","NCK2","SDCBP"],
    "Androgen Apocrine": ["AR","MSLN"],
}

# ── Build tables ──
rows, summary = [], []
for axis, genes in AXES.items():
    genes_clean = [str(g).strip().upper() for g in genes if str(g).strip()]
    in_graph = sorted(g for g in genes_clean if g in graph_genes)
    missing = sorted(g for g in genes_clean if g not in graph_genes)
    for g in in_graph:
        rows.append({"axis": axis, "gene": g, "in_graph": 1})
    summary.append({
        "axis": axis, "n_curated": len(genes_clean),
        "n_in_graph": len(in_graph),
        "coverage": len(in_graph) / max(1, len(genes_clean)),
        "genes_in_graph": ";".join(in_graph),
        "genes_missing": ";".join(missing),
    })
    print(f"[{axis}] curated={len(genes_clean)} | in_graph={len(in_graph)} | missing={len(missing)}")

df_long = pd.DataFrame(rows).sort_values(["axis", "gene"]).reset_index(drop=True) if rows else pd.DataFrame(columns=["axis","gene","in_graph"])
df_summary = pd.DataFrame(summary).sort_values("axis").reset_index(drop=True)

import os; os.makedirs("./Results", exist_ok=True)
df_long.to_csv("./Results/supp_axis_gene_membership.csv", index=False)
df_summary.to_csv("./Results/supp_axis_gene_summary.csv", index=False)

print("\nResumen:")
display(df_summary[["axis", "n_curated", "n_in_graph", "coverage"]])


# Fin